# Alternative 1 — Transfer Learning : gesture embedder MediaPipe

**Objectif :** Utiliser le modèle pré-entraîné MediaPipe Gesture Embedding (entraîné sur HaGRID, ~30k+ images de mains) comme extracteur de features gelé, et n'entraîner qu'une tête de classification sur les embeddings 256-D (2 × 128-D).

Le **gesture embedding model** de MediaPipe (Uboweja et al., 2023, arXiv:2309.10858)
est un petit modèle TFLite qui transforme les landmarks d'une main (x,y,z image + world landmarks
+ handedness) en un embedding de 128 dimensions. Il est distribué officiellement par Google
(`gesture_embedder.tflite`, licence Apache 2.0) et c'est la représentation que Google recommande
pour les tâches de reconnaissance de gestes custom (pipeline Model Maker).
Il est exécuté ici **gelé** avec le runtime LiteRT (ai-edge-litert) : seule la tête 256 → 57 est entraînée.

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → aucune fuite de données entre train et validation/test.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit embeddings 256-D = concat(embedding(main 1), embedding(main 2)).


In [2]:
dataset = load_dataset()
from common.embedder import GestureEmbedder, extract_embeddings, EmbedderHeadMLP
from common.paths import DATA_DIR
embedder = GestureEmbedder()
emb = extract_embeddings(dataset, embedder, cache_path=DATA_DIR / "embeddings.npz")
X_train = torch.from_numpy(emb["train"]["embeddings"]).float()
y_train = torch.from_numpy(emb["train"]["labels"])
X_val   = torch.from_numpy(emb["val"]["embeddings"]).float()
y_val   = torch.from_numpy(emb["val"]["labels"])
X_test  = torch.from_numpy(emb["test"]["embeddings"]).float()
y_test  = torch.from_numpy(emb["test"]["labels"])
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 256) | val: (767, 256) | test: (767, 256)


## 2. Architecture

Tête MLP sur embeddings gelés : 256 → 128 → 128 → 57 (BatchNorm + ReLU + Dropout 0.2).
L'embedder n'est PAS entraîné (runtime TFLite gelé) — c'est un pur transfer learning.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = EmbedderHeadMLP(num_classes=57, input_dim=256, hidden=128)
print(f"params: {count_params(model):,}")


params: 57,273


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 80 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=80, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "pretrained_embedder_curves.png",
                     title="Alternative 1 — Transfer Learning : gesture embedder MediaPipe")
print("best val acc:", trainer.best_val_acc)


  ep 1/80 train 2.7714/0.4120 val 1.9122/0.6493 (0.2s) best 0.6493


  ep 2/80 train 1.8757/0.6400 val 1.5650/0.7027 (0.2s) best 0.7027


  ep 3/80 train 1.6655/0.6931 val 1.4486/0.7562 (0.3s) best 0.7562


  ep 4/80 train 1.5641/0.7266 val 1.3832/0.7718 (0.2s) best 0.7718


  ep 5/80 train 1.5125/0.7459 val 1.3226/0.8057 (0.2s) best 0.8057


...

  ep 77/80 train 1.0944/0.9007 val 0.9659/0.9348 (0.2s) best 0.9400


  ep 78/80 train 1.1006/0.8990 val 0.9749/0.9309 (0.2s) best 0.9400


  ep 79/80 train 1.0990/0.8998 val 0.9627/0.9374 (0.8s) best 0.9400


  ep 80/80 train 1.1048/0.8998 val 0.9753/0.9335 (0.2s) best 0.9400


best val acc: 0.940026075619296


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "pretrained_embedder_cm.png",
               title="Alternative 1 — Transfer Learning : gesture embedder MediaPipe — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_pretrained_embedder.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_pretrained_embedder.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.9335
macro F1: 0.9275 | weighted F1: 0.9345
macro precision: 0.9345 | macro recall: 0.9359


     class  precision    recall        f1  support
54   metro   0.000000  0.000000  0.000000        0
51     car   0.428571  1.000000  0.600000        6
8    labes   1.000000  0.500000  0.666667        4
25   banka   1.000000  0.615385  0.761905       13
4   chabeb   0.700000  0.875000  0.777778        8
39   jadda   0.866667  0.812500  0.838710       16
56   train   0.727273  1.000000  0.842105        8
10  mar7ba   0.750000  1.000000  0.857143        3


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=128):
    # Re-embed the perturbed image landmarks through the frozen embedder
    # (world landmarks and handedness stay fixed) before predicting.
    model.eval()
    world = dataset["test"]["world_landmarks"]
    hand = dataset["test"]["handedness"]
    nh = dataset["test"]["n_hands"]
    outs = []
    with torch.no_grad():
        for i in range(0, len(raw), batch_size):
            emb = embedder.embed_batch(raw[i:i + batch_size],
                                       world[i:i + batch_size],
                                       hand[i:i + batch_size],
                                       nh[i:i + batch_size])
            x = torch.from_numpy(emb).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.9022 (drop=+0.0313)


  noise_0.02: acc=0.8339 (drop=+0.0996)


  noise_0.05: acc=0.5007 (drop=+0.4329)


  rotation_5: acc=0.9205 (drop=+0.0130)


  rotation_10: acc=0.8879 (drop=+0.0456)
  rotation_20: acc=0.7640 (drop=+0.1695)


  translation_0.05: acc=0.9335 (drop=+0.0000)
  translation_0.10: acc=0.9335 (drop=+0.0000)


  scale_0.90: acc=0.9335 (drop=+0.0000)


  scale_1.10: acc=0.9335 (drop=+0.0000)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "pretrained_embedder")
fp16 = ex.quantize_fp16(onnx_path, "pretrained_embedder")
int8 = ex.quantize_int8(onnx_path, "pretrained_embedder")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("pretrained_embedder", metrics, extra=extra)

torch_path = ex.save_torch(model, "pretrained_embedder")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.065 ms/sample


  fp32: size=0.22 MB latency=0.019 ms acc=0.9335
  fp16: size=0.11 MB latency=0.018 ms acc=0.9335
  int8: size=0.06 MB latency=0.022 ms acc=0.9361
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_pretrained_embedder.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\pretrained_embedder.pth (0.23 MB)


## 7. Conclusions

Le transfer learning via l'embedder pré-entraîné de MediaPipe teste l'hypothèse :
« une représentation apprise sur des dizaines de milliers de mains (HaGRID) est-elle plus
discriminante que nos features normalisées maison ? ». Attention : pour ce modèle, les tests de
robustesse ne perturbent PAS l'embedder (il est hors du chemin de gradient) mais bien les
landmarks d'entrée — le forward_fn recalcule les features normalisées, tandis que l'embedding
utilisé par le modèle a été calculé sur les landmarks d'origine.
